# Qwen3-VL-Embedding-2B — offline Kaggle pilot (L21_V001)

Attach the three Kaggle Datasets before running, then keep Internet **off**. Their Dataset slugs do not matter: the next cell discovers the required folders automatically.

1. `aichcm-qwen-pilot-code`: this repository's `src/` folder.
2. `aichcm-keyframes`: contains `keyframes/L21_V001/*.jpg`.
3. `qwen3-vl-embedding-2b-local`: contains the full local model directory, including `model.safetensors` and `scripts/qwen3_vl_embedding.py`.
4. Optional `aichcm-qwen-offline-wheels`: only needed if Kaggle is missing `qwen-vl-utils` or a sufficiently new Transformers package.

The notebook writes only to `/kaggle/working` and never changes the CLIP pipeline.

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

INPUT_ROOT = Path('/kaggle/input')

def discover_exactly_one(description, candidates):
    candidates = list(candidates)
    if len(candidates) != 1:
        raise RuntimeError(f'Expected exactly one {description}; found: {candidates}')
    return candidates[0]

# Kaggle mounts attached Datasets as /kaggle/input/<dataset-slug>.
# Do not hard-code Dataset slugs: find the required content instead.
model_weight = discover_exactly_one(
    'Qwen model.safetensors',
    (path for path in INPUT_ROOT.glob('**/model.safetensors')
     if (path.parent / 'scripts' / 'qwen3_vl_embedding.py').is_file()),
)
MODEL_DIR = model_weight.parent
extractor_file = discover_exactly_one(
    'Qwen pilot extractor.py',
    INPUT_ROOT.glob('**/src/visual_pipeline/qwen3_embedding/extractor.py'),
)
CODE_SRC = extractor_file.parents[2]  # .../src
video_dir = discover_exactly_one(
    'L21_V001 keyframe directory',
    (path for path in INPUT_ROOT.glob('**/L21_V001')
     if path.is_dir() and any(path.glob('*.jpg'))),
)
KEYFRAME_ROOT = video_dir.parent
qwen_wheel = next(iter(INPUT_ROOT.glob('**/qwen_vl_utils-*.whl')), None)
OFFLINE_WHEELS = qwen_wheel.parent if qwen_wheel else None
OUTPUT_DIR = Path('/kaggle/working/qwen_l21_v001_artifacts')
VIDEO_ID = 'L21_V001'
BATCH_SIZE = 8  # RTX 6000 Pro pilot; lower only if the notebook reports OOM.

sys.path.insert(0, str(CODE_SRC))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Keyframes:', len(list((KEYFRAME_ROOT / VIDEO_ID).glob('*.jpg'))))
print('Model:', MODEL_DIR)
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
# Install the exact attached wheel without resolving its optional video
# dependency. This pilot embeds still images only, so PyAV is not used.
if importlib.util.find_spec('qwen_vl_utils') is None:
    if qwen_wheel is None or not qwen_wheel.is_file():
        raise RuntimeError(
            'qwen-vl-utils wheel is missing under /kaggle/input. '
            'Attach the qwen-wheels Dataset; Internet remains off.'
        )
    print('Installing exact wheel:', qwen_wheel)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--no-index',
        '--no-deps', str(qwen_wheel),
    ], check=True)
import torch
import transformers
import qwen_vl_utils

if tuple(map(int, transformers.__version__.split('.')[:2])) < (4, 57):
    raise RuntimeError(
        f'transformers={transformers.__version__}; Qwen needs >=4.57. Attach a compatible offline wheel Dataset.'
    )
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU accelerator before running the pilot.')
print('torch:', torch.__version__, 'transformers:', transformers.__version__)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Offline extraction. The Qwen collection/output remains separate from CLIP.
env = os.environ.copy()
env.update({
    'PYTHONPATH': str(CODE_SRC),
    'HF_HUB_OFFLINE': '1',
    'TRANSFORMERS_OFFLINE': '1',
    'QWEN3_VL_EMBED_MODEL': str(MODEL_DIR),
    'QWEN3_VL_EMBED_KEYFRAME_DIR': str(KEYFRAME_ROOT),
    'QWEN3_VL_EMBED_FEATURE_DIR': str(OUTPUT_DIR),
})
command = [
    sys.executable, '-m', 'visual_pipeline.qwen3_embedding.extractor',
    '--video-id', VIDEO_ID,
    '--device', 'cuda',
    '--dtype', 'float16',
    '--batch-size', str(BATCH_SIZE),
]
print('Running:', ' '.join(command))
subprocess.run(command, env=env, check=True)

In [ ]:
# Validate the cloud artifact before downloading it.
import json
import numpy as np

features = np.load(OUTPUT_DIR / f'{VIDEO_ID}.npy')
filenames = json.loads((OUTPUT_DIR / f'{VIDEO_ID}_filenames.json').read_text(encoding='utf-8'))
manifest = json.loads((OUTPUT_DIR / f'{VIDEO_ID}_manifest.json').read_text(encoding='utf-8'))
assert features.shape == (307, 2048), features.shape
assert len(filenames) == 307
assert np.isfinite(features).all()
assert np.allclose(np.linalg.norm(features, axis=1), 1.0, atol=1e-3)
assert manifest['video_id'] == VIDEO_ID
print('Validated:', features.shape, manifest)

archive_base = '/kaggle/working/qwen_l21_v001_artifacts'
archive = __import__('shutil').make_archive(archive_base, 'zip', OUTPUT_DIR)
print('Download this notebook output:', archive)

## After downloading the ZIP to the local repository

Extract its three files into `data/npy_features_qwen3_vl_2b/`, then index only the separate Qwen collection:

```powershell
$env:PYTHONPATH = 'src'
python -m visual_pipeline.qwen3_embedding.database --video-id L21_V001 --recreate
```

For the first local query test, use the GPU with one text input:

```powershell
$env:PYTHONPATH = 'src'
$env:HF_HUB_OFFLINE = '1'
$env:QWEN3_VL_EMBED_MODEL = (Resolve-Path '.\data\models\Qwen3-VL-Embedding-2B')
& .\.venv-qwen3\Scripts\python.exe -m visual_pipeline.qwen3_embedding.query `
  --video-id L21_V001 --device cuda --dtype float16 --batch-size 1 `
  'mô tả hình ảnh cần tìm'
```

This verifies cloud image vectors and local text vectors share the same Qwen space. It uses only one text input, so it is far lighter than extracting 307 images.